# Exercise 7 - Regression Analysis
AVCAD 2025/2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# load data
df = pd.read_csv('EFIplus_medit.zip', compression='zip', sep=';')
print(df.shape)
df.head()

In [ ]:
# create species richness column
# sum all species presence/absence columns (0/1)
# species columns start at 'Abramis brama'
first_sp = df.columns.get_loc('Abramis brama')
last_sp = df.columns.get_loc('Iberochondrostoma_sp') + 1

df['species_richness'] = df.iloc[:, first_sp:last_sp].sum(axis=1)

print('number of species columns:', last_sp - first_sp)
df['species_richness'].describe()

## Q1 - check distributions and apply transformations

before running regression we need to check if the variables are approximately normally distributed. parametric regression assumes normality of residuals and highly skewed predictors can violate this. we check visually with histograms and Q-Q plots

In [ ]:
env_vars = ['Altitude', 'Actual_river_slope', 'Elevation_mean_catch',
            'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul']

# check skewness
for var in env_vars:
    print(f'{var}: skewness = {df[var].dropna().skew():.3f}')

In [ ]:
# histogram + Q-Q plot for each variable
fig, axes = plt.subplots(7, 2, figsize=(13, 28))
fig.suptitle('Q1 - Visual check of distributions', fontsize=13, fontweight='bold', y=1.01)

for i, var in enumerate(env_vars):
    data = df[var].dropna()
    sk = data.skew()
    needs_transf = var in ['Altitude', 'Actual_river_slope', 'Elevation_mean_catch']
    col = '#E24B4A' if needs_transf else '#1D9E75'
    label = 'transformation required' if needs_transf else 'ok'

    axes[i, 0].hist(data, bins=50, color=col, edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{var} | skewness = {sk:.2f} | {label}', fontsize=9)
    axes[i, 0].set_xlabel('Value')
    axes[i, 0].set_ylabel('Count')
    for spine in ['top', 'right']:
        axes[i, 0].spines[spine].set_visible(False)

    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    axes[i, 1].scatter(osm, osr, alpha=0.3, s=5, color=col)
    axes[i, 1].plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2)
    axes[i, 1].set_title(f'Q-Q plot | r = {r:.3f}', fontsize=9)
    axes[i, 1].set_xlabel('Theoretical quantiles')
    axes[i, 1].set_ylabel('Sample quantiles')
    for spine in ['top', 'right']:
        axes[i, 1].spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

based on the histograms and Q-Q plots:
- **Altitude** (skew=1.0): right skewed, needs transformation
- **Actual_river_slope** (skew=10.3): very strongly right skewed - most rivers have low slope but some extreme values push the distribution. definitely needs transformation
- **Elevation_mean_catch** (skew=0.85): moderate right skew, needs transformation
- **prec_ann_catch** (skew=0.058): nearly symmetric, no transformation needed
- **temp_ann, temp_jan, temp_jul**: skewness is low and Q-Q plots look reasonable, no transformation needed

In [ ]:
# apply transformations
# log1p (= log(x+1)) for Altitude and slope - handles zeros safely since log(0) is undefined
# sqrt for Elevation - gentler transformation for moderate skew

df['log1p_Altitude'] = np.log1p(df['Altitude'])
df['log1p_Actual_river_slope'] = np.log1p(df['Actual_river_slope'])
df['sqrt_Elevation_mean_catch'] = np.sqrt(df['Elevation_mean_catch'])

print('skewness after transformation:')
print('Altitude:', round(df['log1p_Altitude'].dropna().skew(), 3))
print('Slope:', round(df['log1p_Actual_river_slope'].dropna().skew(), 3))
print('Elevation:', round(df['sqrt_Elevation_mean_catch'].dropna().skew(), 3))

In [ ]:
# check distributions after transformation
transf_vars = [
    ('Altitude', 'log1p_Altitude', 'log(Altitude+1)'),
    ('Actual_river_slope', 'log1p_Actual_river_slope', 'log(Slope+1)'),
    ('Elevation_mean_catch', 'sqrt_Elevation_mean_catch', 'sqrt(Elevation)')
]

fig2, axes2 = plt.subplots(3, 2, figsize=(13, 14))
fig2.suptitle('Q1 - Distributions after transformation', fontsize=13, fontweight='bold')

for i, (orig, trans, label) in enumerate(transf_vars):
    data_t = df[trans].dropna()
    sk_before = df[orig].dropna().skew()
    sk_after = data_t.skew()

    axes2[i, 0].hist(data_t, bins=50, color='#1D9E75', edgecolor='white', alpha=0.8)
    axes2[i, 0].set_title(f'{label} | skewness: {sk_before:.2f} -> {sk_after:.2f}', fontsize=10)
    axes2[i, 0].set_xlabel('Transformed value')
    axes2[i, 0].set_ylabel('Count')
    for spine in ['top', 'right']:
        axes2[i, 0].spines[spine].set_visible(False)

    (osm, osr), (slope, intercept, r) = stats.probplot(data_t, dist='norm')
    axes2[i, 1].scatter(osm, osr, alpha=0.3, s=5, color='#1D9E75')
    axes2[i, 1].plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2)
    axes2[i, 1].set_title(f'Q-Q after transformation | r = {r:.3f}', fontsize=10)
    axes2[i, 1].set_xlabel('Theoretical quantiles')
    axes2[i, 1].set_ylabel('Sample quantiles')
    for spine in ['top', 'right']:
        axes2[i, 1].spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

transformations worked well - especially river slope which went from skew=10.3 to 0.09

In [ ]:
# simple linear regressions - species richness vs each predictor (after transformation)

predictors = {
    'log(Altitude+1)': 'log1p_Altitude',
    'log(Slope+1)': 'log1p_Actual_river_slope',
    'sqrt(Elevation)': 'sqrt_Elevation_mean_catch',
    'prec_ann_catch': 'prec_ann_catch',
    'temp_ann': 'temp_ann',
    'temp_jan': 'temp_jan',
    'temp_jul': 'temp_jul',
}

results = {}
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
fig.suptitle('Q1 - Simple linear regression: species richness ~ each predictor',
             fontsize=13, fontweight='bold')

colors = ['#3B8BD4', '#1D9E75', '#E24B4A', '#F39C12', '#9B59B6', '#E67E22', '#1ABC9C']

for idx, (name, col) in enumerate(predictors.items()):
    ax = axes.flat[idx]
    sub = df[[col, 'species_richness']].dropna()
    x = sub[col].values
    y = sub['species_richness'].values

    slope_reg, intercept, r, p, se = stats.linregress(x, y)
    r2 = r**2
    n = len(x)
    y_pred = intercept + slope_reg * x
    SS_res = np.sum((y - y_pred)**2)
    SS_tot = np.sum((y - np.mean(y))**2)
    F = (SS_tot - SS_res) / (SS_res / (n - 2))

    results[name] = {'n': n, 'b0': intercept, 'b1': slope_reg, 'r2': r2, 'F': F, 'p': p}

    ax.scatter(x, y, alpha=0.15, s=8, color=colors[idx])
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, intercept + slope_reg * x_line, color='red', lw=2)
    ax.set_xlabel(name, fontsize=9)
    ax.set_ylabel('Species richness', fontsize=9)
    p_str = 'p < 0.001' if p < 0.001 else f'p = {p:.3f}'
    ax.set_title(f'b1={slope_reg:.3f} | R2={r2:.3f} | F={F:.1f} | {p_str}', fontsize=8.5)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

for ax in list(axes.flat)[7:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# summary table
print(f'{"Predictor":<25} {"n":>6} {"b0":>8} {"b1":>8} {"R2":>8} {"F":>10} {"p":>12}')
print('-' * 82)
for name, r in results.items():
    p_str = '< 0.001' if r['p'] < 0.001 else f"{r['p']:.4f}"
    print(f"{name:<25} {r['n']:>6} {r['b0']:>8.3f} {r['b1']:>8.4f} {r['r2']:>8.4f} {r['F']:>10.1f} {p_str:>12}")

all predictors are statistically significant (p < 0.001). log(Slope+1) has the highest R2 (0.134) followed by log(Altitude+1) (0.112) - both negative, meaning higher and steeper rivers have fewer species. temperature variables show positive relationships. precipitation has the lowest R2 (0.023)

## Q2 - Multiple linear regression

now we fit one model with all predictors at once. coefficients now represent the **partial effect** of each predictor while controlling for all the others - this is different from the univariate models above

In [ ]:
pred_cols = ['log1p_Altitude', 'log1p_Actual_river_slope', 'sqrt_Elevation_mean_catch',
             'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul']
pred_labels = ['log(Altitude+1)', 'log(Slope+1)', 'sqrt(Elevation)',
               'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul']

# prepare data - drop rows with any NAs in these columns
data = df[pred_cols + ['species_richness']].dropna()
X = data[pred_cols].copy()
y = data['species_richness']

# fit full multiple regression
X_sm = sm.add_constant(X)
model_full = sm.OLS(y, X_sm).fit()
print(model_full.summary())

In [ ]:
# partial dependence plots
# partial residuals = residuals + contribution of each predictor
# this isolates the effect of each variable controlling for the others

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Q2 - Partial dependence plots (full multiple regression model)',
             fontsize=12, fontweight='bold')

colors = ['#3B8BD4', '#1D9E75', '#E24B4A', '#F39C12', '#9B59B6', '#E67E22', '#1ABC9C']

for idx, (col, label) in enumerate(zip(pred_cols, pred_labels)):
    ax = axes.flat[idx]
    partial_resid = model_full.resid + model_full.params[col] * data[col]
    ax.scatter(data[col], partial_resid, alpha=0.1, s=6, color=colors[idx])
    x_range = np.linspace(data[col].min(), data[col].max(), 200)
    ax.plot(x_range, model_full.params['const'] + model_full.params[col] * x_range,
            color='red', lw=2)
    b1 = model_full.params[col]
    p = model_full.pvalues[col]
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_title(f'{label}\nb1 = {b1:.3f} {sig}', fontsize=9)
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel('Partial residuals', fontsize=8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

axes.flat[7].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# compare univariate vs multiple coefficients
univ_b1 = [-0.7890, -1.0269, -0.0857, -0.0015, 0.4060, 0.2192, 0.3370]
multi_b1 = [model_full.params[c] for c in pred_cols]

print('Univariate vs multiple regression coefficients:')
print(f'{"Predictor":<25} {"Univariate b1":>15} {"Multiple b1":>15} {"Change %":>10}')
print('-' * 70)
for label, b_uni, b_multi in zip(pred_labels, univ_b1, multi_b1):
    change = ((b_multi - b_uni) / abs(b_uni)) * 100
    print(f'{label:<25} {b_uni:>15.4f} {b_multi:>15.4f} {change:>9.1f}%')

print(f'\nFull model R2: {model_full.rsquared:.4f}')
print(f'AIC: {model_full.aic:.2f}')

some coefficients changed a lot between univariate and multiple models - this happens because correlated predictors were absorbing each other's effects in the univariate models. the multiple model R2 (0.25) is higher than any single predictor alone, meaning that combining all predictors explains more variation in species richness

## Q3 - Multicollinearity and parsimonious model

when predictors are strongly correlated with each other the coefficient estimates become unreliable. we use VIF (Variance Inflation Factor) to check this:
- VIF <= 5: ok
- 5 < VIF <= 10: moderate concern  
- VIF > 10: severe multicollinearity

In [ ]:
# compute VIF for each predictor
X_vif = sm.add_constant(X)
vifs = [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(pred_cols))]

print('VIF results:')
for label, vif in zip(pred_labels, vifs):
    flag = ' <- SEVERE (>10)' if vif > 10 else ' <- moderate (>5)' if vif > 5 else ''
    print(f'  {label:<25}: {vif:.1f}{flag}')

In [ ]:
# VIF bar chart
fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#E24B4A' if v > 10 else '#F39C12' if v > 5 else '#1D9E75' for v in vifs]
bars = ax.barh(pred_labels, vifs, color=bar_colors, edgecolor='white', height=0.6)
ax.axvline(x=5, color='orange', lw=1.5, linestyle='--', label='VIF = 5 (moderate)')
ax.axvline(x=10, color='red', lw=1.5, linestyle='--', label='VIF = 10 (severe)')
ax.set_xlabel('Variance Inflation Factor (VIF)')
ax.set_title('Q3 - VIF per predictor', fontweight='bold')
ax.legend(fontsize=9)
for bar, vif in zip(bars, vifs):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{vif:.1f}', va='center', fontsize=9)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

temp_ann (VIF=56.5), temp_jan (VIF=27.6) and temp_jul (VIF=18.9) all have severe multicollinearity - which makes sense because annual, january and july temperatures are obviously strongly correlated with each other.

solution: keep only temp_ann as representative of the three temperature variables and remove temp_jan and temp_jul. this follows the parsimony principle - prefer simpler models when predictors are redundant

In [ ]:
# parsimonious model - remove temp_jan and temp_jul
pred_cols_pars = ['log1p_Altitude', 'log1p_Actual_river_slope', 'sqrt_Elevation_mean_catch',
                  'prec_ann_catch', 'temp_ann']
pred_labels_pars = ['log(Altitude+1)', 'log(Slope+1)', 'sqrt(Elevation)',
                    'prec_ann_catch', 'temp_ann']

X_pars = data[pred_cols_pars].copy()
X_pars_sm = sm.add_constant(X_pars)
model_pars = sm.OLS(y, X_pars_sm).fit()

print('Parsimonious model (5 predictors):')
print(f'R2:          {model_pars.rsquared:.4f}')
print(f'R2 adjusted: {model_pars.rsquared_adj:.4f}')
print(f'AIC:         {model_pars.aic:.2f}')
print('\nCoefficients:')
for col, label in zip(pred_cols_pars, pred_labels_pars):
    p = model_pars.pvalues[col]
    p_s = '< 0.001' if p < 0.001 else f'{p:.4f}'
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {label:<25}: b1 = {model_pars.params[col]:>8.4f}   p {p_s}  {sig}')

In [ ]:
# coefficient comparison + residual diagnostics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Q3 - Full vs parsimonious model', fontsize=12, fontweight='bold')

b1_full = [model_full.params[c] for c in pred_cols_pars]
b1_pars = [model_pars.params[c] for c in pred_cols_pars]
x_pos = np.arange(len(pred_labels_pars))
width = 0.35

axes[0].bar(x_pos - width/2, b1_full, width,
            label=f'Full model (7 pred) R2={model_full.rsquared:.3f} AIC={model_full.aic:.0f}',
            color='#E24B4A', alpha=0.8, edgecolor='white')
axes[0].bar(x_pos + width/2, b1_pars, width,
            label=f'Parsimonious (5 pred) R2={model_pars.rsquared:.3f} AIC={model_pars.aic:.0f}',
            color='#1D9E75', alpha=0.8, edgecolor='white')
axes[0].axhline(y=0, color='black', lw=0.8)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(pred_labels_pars, rotation=25, ha='right', fontsize=9)
axes[0].set_ylabel('Regression coefficient b1')
axes[0].set_title('Coefficient comparison')
axes[0].legend(fontsize=8)
for spine in ['top', 'right']:
    axes[0].spines[spine].set_visible(False)

# residuals vs fitted values
fitted = model_pars.fittedvalues
resid = model_pars.resid
axes[1].scatter(fitted, resid, alpha=0.15, s=8, color='#3B8BD4')
axes[1].axhline(y=0, color='red', lw=1.5, linestyle='--')
axes[1].set_xlabel('Fitted values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs fitted (parsimonious model)')
for spine in ['top', 'right']:
    axes[1].spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# final model comparison
print('Model comparison:')
print(f'Full model (7 pred)  : R2={model_full.rsquared:.4f}, AIC={model_full.aic:.1f}')
print(f'Parsimonious (5 pred): R2={model_pars.rsquared:.4f}, AIC={model_pars.aic:.1f}')
print(f'Delta AIC: {model_pars.aic - model_full.aic:.1f}')

the full model has lower AIC (22404 vs 22743) but this is misleading - it keeps three temperature variables with severe multicollinearity (VIF > 10) which makes the coefficients unreliable.

the parsimonious model loses some R2 (0.197 vs 0.253) but produces more stable and interpretable coefficients. removing temp_jan and temp_jul does not lose important information since temp_ann already captures the overall temperature effect.

the most important predictors of species richness are river slope and altitude (both negative - steeper and higher rivers have fewer species) and temperature (positive - warmer sites tend to have more species in this mediterranean dataset)